In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import os



In [ ]:

path = kagglehub.dataset_download("smid80/weatherww2")
print("Path to dataset files:", path)

100%|██████████| 1.65M/1.65M [00:00<00:00, 2.88MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/smid80/weatherww2/versions/1


In [ ]:
files=os.listdir(path)
print(files)

['Weather Station Locations.csv', 'Summary of Weather.csv']


In [ ]:
df = pd.read_csv(os.path.join(path, 'Summary of Weather.csv'), low_memory=False)
#Veri setindeki ilk 5 kayıt içeriği
print(df.head())

     STA      Date Precip  WindGustSpd    MaxTemp    MinTemp   MeanTemp  \
0  10001  1942-7-1  1.016          NaN  25.555556  22.222222  23.888889   
1  10001  1942-7-2      0          NaN  28.888889  21.666667  25.555556   
2  10001  1942-7-3   2.54          NaN  26.111111  22.222222  24.444444   
3  10001  1942-7-4   2.54          NaN  26.666667  22.222222  24.444444   
4  10001  1942-7-5      0          NaN  26.666667  21.666667  24.444444   

  Snowfall PoorWeather  YR  ...  FB  FTI ITH  PGT  TSHDSBRSGF  SD3  RHX  RHN  \
0        0         NaN  42  ... NaN  NaN NaN  NaN         NaN  NaN  NaN  NaN   
1        0         NaN  42  ... NaN  NaN NaN  NaN         NaN  NaN  NaN  NaN   
2        0         NaN  42  ... NaN  NaN NaN  NaN         NaN  NaN  NaN  NaN   
3        0         NaN  42  ... NaN  NaN NaN  NaN         NaN  NaN  NaN  NaN   
4        0         NaN  42  ... NaN  NaN NaN  NaN         NaN  NaN  NaN  NaN   

  RVG  WTE  
0 NaN  NaN  
1 NaN  NaN  
2 NaN  NaN  
3 NaN  NaN  
4 N

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119040 entries, 0 to 119039
Data columns (total 31 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   STA          119040 non-null  int64  
 1   Date         119040 non-null  object 
 2   Precip       119040 non-null  object 
 3   WindGustSpd  532 non-null     float64
 4   MaxTemp      119040 non-null  float64
 5   MinTemp      119040 non-null  float64
 6   MeanTemp     119040 non-null  float64
 7   Snowfall     117877 non-null  object 
 8   PoorWeather  34237 non-null   object 
 9   YR           119040 non-null  int64  
 10  MO           119040 non-null  int64  
 11  DA           119040 non-null  int64  
 12  PRCP         117108 non-null  object 
 13  DR           533 non-null     float64
 14  SPD          532 non-null     float64
 15  MAX          118566 non-null  float64
 16  MIN          118572 non-null  float64
 17  MEA          118542 non-null  float64
 18  SNF          117877 non-

In [ ]:
#NaN değerlerin adetini bulmayı sağlar.
print(df.isnull().sum())

STA                 0
Date                0
Precip              0
WindGustSpd    118508
MaxTemp             0
MinTemp             0
MeanTemp            0
Snowfall         1163
PoorWeather     84803
YR                  0
MO                  0
DA                  0
PRCP             1932
DR             118507
SPD            118508
MAX               474
MIN               468
MEA               498
SNF              1163
SND            113477
FT             119040
FB             119040
FTI            119040
ITH            119040
PGT            118515
TSHDSBRSGF      84803
SD3            119040
RHX            119040
RHN            119040
RVG            119040
WTE            119040
dtype: int64


In [ ]:
#Bu sütunların kalıcı olarak silinmesi sağlanmıştır.
df.drop([ "WindGustSpd","DR","SPD","SND","PGT",
    "FT","FB","FTI","ITH","SD3","RHX","RHN","RVG","WTE",
    "PoorWeather","TSHDSBRSGF","PRCP","SNF","MAX","MIN","MEA"],axis=1,inplace=True)

In [11]:
 #Hangi aylarda ne kadar snowfall verisi boştur.
print(df[df['Snowfall'].isna()]['MO'].value_counts().sort_index())

MO
1     140
2     133
3     109
4     146
5     145
6      88
7       8
8       1
9      10
10     91
11    163
12    129
Name: count, dtype: int64


In [12]:
#Kolonların listesi
print(df.columns.tolist())

['STA', 'Date', 'Precip', 'MaxTemp', 'MinTemp', 'MeanTemp', 'Snowfall', 'YR', 'MO', 'DA']


In [13]:
df["Snowfall"]=pd.to_numeric(df["Snowfall"],errors="coerce")

In [14]:
# Snowfall sütunundaki eksik (NaN) değerler,
# her bir istasyon (STA) ve ay (MO) bazında gruplanarak,
# ilgili grubun medyan (median) değeri ile doldurulmuştur.
df['Snowfall'] = df.groupby(['STA', 'MO'])['Snowfall'].transform(
    lambda x: x.fillna(x.median())
)

In [16]:
# Snowfall sütunundaki eksik değerler, her ay (MO) için gruplanarak,
# ilgili ayın medyan değeri ile doldurulmuştur.
df['Snowfall'] = df.groupby('MO')['Snowfall'].transform(
    lambda x: x.fillna(x.median())
)

In [17]:
#NaN değer kontrol
print(df.isnull().sum())

STA         0
Date        0
Precip      0
MaxTemp     0
MinTemp     0
MeanTemp    0
Snowfall    0
YR          0
MO          0
DA          0
dtype: int64


In [18]:
#Precip sütunundaki değerlerin başındaki ve sonundaki boşluklar kaldırıldı (strip)
df['Precip'] = df['Precip'].str.strip()

In [19]:
#'T' (trace/çok az yağış) değerleri 0 ile değiştirildi, çünkü bu değerler ihmal edilebilir miktarı temsil eder.
df['Precip'] = df['Precip'].replace('T', '0')

In [20]:
#Tüm değerler pd.to_numeric ile sayısal tipe dönüştürüldü.
#Hatalı veya dönüştürülemeyen değerler NaN olarak işaretlendi (errors='coerce').
df['Precip'] = pd.to_numeric(df['Precip'], errors='coerce')

In [21]:
print(df['Precip'].isna().sum())

0


In [22]:
df.drop(["STA"],axis=1,inplace=True)

In [23]:
X=df.drop(columns=["MeanTemp","Date"],axis=1) #İndependent features (STA, Precip, MaxTemp, MinTemp, Snowfall, YR, MO, DA)
y=df["MeanTemp"] #Dependent feature(MeanTemp)

In [24]:
print(X)
print(y)

        Precip    MaxTemp    MinTemp  Snowfall  YR  MO  DA
0        1.016  25.555556  22.222222       0.0  42   7   1
1        0.000  28.888889  21.666667       0.0  42   7   2
2        2.540  26.111111  22.222222       0.0  42   7   3
3        2.540  26.666667  22.222222       0.0  42   7   4
4        0.000  26.666667  21.666667       0.0  42   7   5
...        ...        ...        ...       ...  ..  ..  ..
119035   0.000  28.333333  18.333333       0.0  45  12  27
119036   9.906  29.444444  18.333333       0.0  45  12  28
119037   0.000  28.333333  18.333333       0.0  45  12  29
119038   0.000  28.333333  18.333333       0.0  45  12  30
119039   0.000  29.444444  17.222222       0.0  45  12  31

[119040 rows x 7 columns]
0         23.888889
1         25.555556
2         24.444444
3         24.444444
4         24.444444
            ...    
119035    23.333333
119036    23.888889
119037    23.333333
119038    23.333333
119039    23.333333
Name: MeanTemp, Length: 119040, dtype: float6

In [25]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [26]:
#Scale işlemleri için gerekli
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()

In [27]:
X_train_scaled=sc.fit_transform(X_train)
X_test_scaled=sc.transform(X_test)

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [29]:
lr=LinearRegression()
lr.fit(X_train_scaled,y_train)
y_pred=lr.predict(X_test_scaled)

In [30]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("RMSE:", rmse)
print("R2:", r2)
print("MAE:", mae)


RMSE: 0.4401423350812684
R2: 0.9971482525969805
MAE: 0.1651748360202977


In [31]:
from sklearn.linear_model import Ridge
rd=Ridge(alpha=1.0)
rd.fit(X_train_scaled,y_train)
y_pred=rd.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("RMSE:", rmse)
print("R2:", r2)
print("MAE:", mae)


RMSE: 0.44014063724952623
R2: 0.9971482745979449
MAE: 0.16516400663211844


In [32]:
from sklearn.linear_model import Lasso
ls=Lasso(alpha=1.0)

ls.fit(X_train_scaled,y_train)
y_pred=ls.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("RMSE:", rmse)
print("R2:", r2)
print("MAE:", mae)


RMSE: 1.1065017028144446
R2: 0.9819769228572974
MAE: 0.7404101450671609


In [33]:
from sklearn.linear_model import ElasticNet

en=ElasticNet(alpha=0.1,l1_ratio=0.5)
en.fit(X_train_scaled,y_train)
y_pred=en.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("RMSE:", rmse)
print("R2:", r2)
print("MAE:", mae)


RMSE: 0.5081564928181331
R2: 0.9961988087017015
MAE: 0.2594278837920118


In [ ]:
# Modellerin performans özetleri:
# - Lineer ve Ridge: En iyi performans, RMSE düşük, R2 ~0.997
# - Lasso: Hata biraz yüksek, bazı değişkenler sıfırlandı
# - ElasticNet: Dengeli performans, Lineer/Ridge’e yakın
# → Genel olarak veri lineer modellerle iyi açıklanabiliyor.